<a href="https://colab.research.google.com/github/adilsaleem007/Python-Basics-Assignment/blob/main/Assignment7NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1) Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from tensorflow.keras.utils import to_categorical

# 2) Load Dataset

In [2]:
df = pd.read_csv('/content/judge-1377884607_tweet_product_company.csv', encoding='latin-1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


# 3) Data Preprocessing

In [3]:
#Drop unwanted column
df = df.drop(columns=['emotion_in_tweet_is_directed_at'], errors='ignore')

In [4]:
#Handle missing values
df = df.dropna()

# 4) Text Cleaning

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['clean_text'] = df['tweet_text'].apply(clean_text)

# 5) Encode Labels

In [7]:
le = LabelEncoder()
df['is_there_an_emotion_directed_at_a_brand_or_product'] = le.fit_transform(df['is_there_an_emotion_directed_at_a_brand_or_product'])

y = to_categorical(df['is_there_an_emotion_directed_at_a_brand_or_product'])

# 6) Tokenization & Padding

In [8]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['clean_text'])

X = tokenizer.texts_to_sequences(df['clean_text'])
X = pad_sequences(X, maxlen=50)

# 7) Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 8) LSTM

In [10]:
model = Sequential()
model.add(Embedding(10000, 128, input_length=100))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(4, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


# 9) Train Model

In [11]:
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 40s 154ms/step - accuracy: 0.5914 - loss: 0.9158 - val_accuracy: 0.6449 - val_loss: 0.8311
Epoch 2/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 33s 144ms/step - accuracy: 0.6883 - loss: 0.7425 - val_accuracy: 0.6811 - val_loss: 0.7814
Epoch 3/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 42s 147ms/step - accuracy: 0.7642 - loss: 0.5920 - val_accuracy: 0.6641 - val_loss: 0.8100
Epoch 4/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 36s 155ms/step - accuracy: 0.8009 - loss: 0.4988 - val_accuracy: 0.6872 - val_loss: 0.8598
Epoch 5/5
228/228 ━━━━━━━━━━━━━━━━━━━━ 38s 144ms/step - accuracy: 0.8302 - loss: 0.4314 - val_accuracy: 0.6619 - val_loss: 0.9528


# 10) Evaluate Model

In [12]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.6619 - loss: 0.9528
Accuracy: 0.661902129650116


# 11) Prediction Example

In [13]:
sample = ["I love this Google product"]
sample_seq = tokenizer.texts_to_sequences(sample)
sample_pad = pad_sequences(sample_seq, maxlen=50)

prediction = model.predict(sample_pad)
print(le.inverse_transform([np.argmax(prediction)]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 502ms/step
['Positive emotion']
